# Guided Practice - Python for Data Engineering

This notebook complements `00_intro_python_data_engineering.py`.

Goals:
- Understand ETL phases
- Run a mini ETL flow step by step
- Validate record quality with simple rules

## 1) Define a Small Raw Dataset

In [1]:
raw_records = [
    {"id": 1, "customer": "Alice", "amount": "250.00", "country": "co"},
    {"id": 2, "customer": "Bob", "amount": "-30", "country": "mx"},
    {"id": 3, "customer": "Carol ", "amount": "175.5", "country": "co"},
    {"id": 4, "customer": "", "amount": "invalid", "country": "pe"},
    {"id": 5, "customer": " John ", "amount": "120.00", "country": "cl"},
    {"id": 6, "customer": "  Charlie", "amount": "90.00", "country": "mx"},
]
raw_records

[{'id': 1, 'customer': 'Alice', 'amount': '250.00', 'country': 'co'},
 {'id': 2, 'customer': 'Bob', 'amount': '-30', 'country': 'mx'},
 {'id': 3, 'customer': 'Carol ', 'amount': '175.5', 'country': 'co'},
 {'id': 4, 'customer': '', 'amount': 'invalid', 'country': 'pe'},
 {'id': 5, 'customer': ' John ', 'amount': '120.00', 'country': 'cl'},
 {'id': 6, 'customer': '  Charlie', 'amount': '90.00', 'country': 'mx'}]

## 2) Transform Step
Normalize text fields and cast `amount` to float when possible.

In [2]:
def transform(records):
    transformed = []
    for row in records:
        new_row = row.copy()
        new_row["country"] = new_row["country"].upper()
        new_row["customer"] = new_row["customer"].strip().title()
        try:
            new_row["amount"] = float(new_row["amount"])
        except (TypeError, ValueError):
            new_row["amount"] = None
        transformed.append(new_row)
    return transformed


clean_records = transform(raw_records)
clean_records

[{'id': 1, 'customer': 'Alice', 'amount': 250.0, 'country': 'CO'},
 {'id': 2, 'customer': 'Bob', 'amount': -30.0, 'country': 'MX'},
 {'id': 3, 'customer': 'Carol', 'amount': 175.5, 'country': 'CO'},
 {'id': 4, 'customer': '', 'amount': None, 'country': 'PE'},
 {'id': 5, 'customer': 'John', 'amount': 120.0, 'country': 'CL'},
 {'id': 6, 'customer': 'Charlie', 'amount': 90.0, 'country': 'MX'}]

## 3) Validate Step
Keep records where:
- customer is present
- amount is numeric and positive

In [3]:
def validate(records):
    valid = []
    invalid = []
    for row in records:
        if not row["customer"] or row["amount"] is None or row["amount"] <= 0:
            invalid.append(row)
            continue
        valid.append(row)
    return valid, invalid


valid_records, invalid_records = validate(clean_records)
len(valid_records), len(invalid_records)

(4, 2)

## 4) Load + Summary
In this practice notebook, loading is simulated by printing and summary metrics.

In [4]:
for row in valid_records:
    print(f"LOAD -> id={row['id']}, customer={row['customer']}, amount={row['amount']}, country={row['country']}")

summary = {
    "extracted": len(raw_records),
    "trusted": len(valid_records),
    "rejected": len(invalid_records),
    "total_amount": round(sum(r["amount"] for r in valid_records), 2),
}
summary

LOAD -> id=1, customer=Alice, amount=250.0, country=CO
LOAD -> id=3, customer=Carol, amount=175.5, country=CO
LOAD -> id=5, customer=John, amount=120.0, country=CL
LOAD -> id=6, customer=Charlie, amount=90.0, country=MX


{'extracted': 6, 'trusted': 4, 'rejected': 2, 'total_amount': 635.5}

## 5) Student Challenge
Add one rule: keep only records from country `CO` and recompute summary.